In [2]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Filtrar SIN GESTION
# df_filtrado = df_cen_sae[
#     ~df_cen_sae["Mejor_Descripcion_Telefono"].str.contains("SIN GESTION", na=False)
# ]
df_filtrado = df_cen_sae[
    # (~df_cen_sae["cliTelefono"].str.contains("SIN GESTION", na=False)) &
    (df_cen_sae["tipo_telf"].isin(['BBDD CEL01','BBDD CEL03','BBDD CEL02']))&
    (df_cen_sae["PROPENSION"].isin(['ALTO','BAJO']))
]

# Conteo ordenado
conteo = df_filtrado["mejor_descripcion_cli"] \
            .value_counts() \
            .sort_values(ascending=False)

# % acumulado
porc_acum = conteo.cumsum() / conteo.sum() * 100

fig, ax1 = plt.subplots(figsize=(12,6))

# 🔵 Barras
bars = ax1.bar(conteo.index, conteo.values, color="steelblue")
ax1.set_ylabel("Cantidad")

# 👉 Etiquetas de cantidad en barras
for bar in bars:
    height = bar.get_height()
    ax1.text(
        bar.get_x() + bar.get_width()/2,
        height,
        f'{int(height)}',
        ha='center',
        va='bottom'
    )

plt.xticks(rotation=45, ha="right")

# 🔴 Línea acumulada
ax2 = ax1.twinx()
ax2.plot(conteo.index, porc_acum, color="red", marker="o")
ax2.set_ylabel("% Acumulado")

# 👉 Etiquetas de % acumulado
for i, txt in enumerate(porc_acum):
    ax2.text(i, txt, f"{txt:.1f}%", color="red", ha="center", va="bottom")

# Línea 80%
ax2.axhline(80, color="green", linestyle="--")

plt.title("Gráfico de Pareto - Mejor_Descripcion_Telefono")
plt.tight_layout()
plt.show()

In [ ]:
# df_lista.write.mode("overwrite").parquet(
# "C:\\Users\\DATA\\Documents\\datos\\01_script\\inicio\\dataset"
# )

# pendiente

In [ ]:
query = "SELECT * FROM CRONOX.dbo.borrar_df_vicidial"
df_cen=get_data_sql(query, params_sa)
overwrite_table_SQL(spark,df_vicidial,'borrar_df_vicidial',server_sa,user_sa,pwd_sa,'CRONOX')
df_vicidial.to_csv("C:\\Users\\DATA\\Documents\\datos\\01_script\\inicio\\borrar_df_vicidial.csv", index=False)

query = "SELECT * FROM CRONOX.dbo.borrar_df_tnumer"
df_cen=get_data_sql(query, params_sa)
df_tnumer.to_csv("C:\\Users\\DATA\\Documents\\datos\\01_script\\inicio\\borrar_df_tnumer1.csv", index=False)

query = "SELECT * FROM CRONOX.dbo.borrar_df_venta"
df_cen=get_data_sql(query, params_sa)
df_venta.to_csv("C:\\Users\\DATA\\Documents\\datos\\01_script\\inicio\\borrar_df_venta.csv", index=False)

query = "SELECT * FROM CRONOX.dbo.borrar_df_base"
df_cen=get_data_00l(query, params_sa)
df_base.to_csv("C:\\Users\\DATA\\Documents\\datos\\01_script\\inicio\\borrar_df_base.csv", index=False)

In [15]:
overwrite_table_SQL(spark,df_lista,'borrar_df_cencosud_prestamo',server_sa,user_sa,pwd_sa,'CRONOX')

In [12]:

df_lista=df_tnumer.join(df_base,['vendor_lead_code'],'left')
df_lista=df_lista.join(df_vicidial,['vendor_lead_code','phone_number'],'left')
df_lista=df_lista.join(df_venta,['vendor_lead_code','title'],'left')
window_spec = Window.partitionBy("indice").orderBy(col("fecha_llamada").desc())
df_lista = df_lista.withColumn("dni_unico", row_number().over(window_spec))

In [13]:
print(df_lista.columns)

['vendor_lead_code', 'title', 'phone_number', 'tipo_telf', 'orden_telf', 'indice', 'tipo_base', 'regimen_laboral', 'producto', 'marca', 'tea', 'retiro_01', 'FEC_PAGO', 'FECHA_ULT_ATM', 'MONTO_ANT', 'FECHA_ULT_COMP', 'MONTO_DESEMBOLSAR', 'RNG_SALDO_TC_ENTRE_LINEA_TOTAL_TC', 'DISP_RETIRO_EFECT_', 'MEJORA_TASA', 'LINEA_SAE', 'PCT_SAE', 'ENTIDAD1', 'DEUDA1', 'ENTIDAD2', 'DEUDA2', 'ENTIDAD3', 'DEUDA3', 'EDAD', 'FRESCURA_TARGET', 'Segmentacion_Montos', 'FLG_MEJORA', 'PROPENSION', 'TIPDOC', 'first_name', 'last_name', 'address1', 'address2', 'address3', 'city', 'province', 'security_phrase', 'email', 'comments', 'CODIGO', 'numero_campana', 'nombre_campana', 'dni_ejecutivo', 'ejecutivo', 'fecha_hora_llamada', 'duracion', 'descripcion_sn', 'list_description', 'list_name', 'fecha_agenda', 'comentarios', 'fecha_llamada', 'trama', 'COD_TIPO', 'TIPO', 'COD_SUB_DESCRIPCION', 'SUB_DESCRIPCION', 'DESCRIPCION', 'COD_BCO', 'PESO', 'DESC_CODCLI', 'n_mejor_tel', 'mejor_descripcin_telf', 'mejor_fecha_llamad

In [14]:
df_lista.count()

87640

In [ ]:



df_pru=df_lista.filter(df_lista.mejor_descripcin_cli.contains( 'CASILLA'))

In [ ]:
window_spec = Window.partitionBy("indice").orderBy(col("fecha_llamada").desc())
df_lista = df_lista.filter(col("dni_unico") == 1).drop('dni_unico')

In [ ]:
df_lista[['LINEA_SAE']].distinct().show()


In [70]:
query = "SELECT * FROM CRONOX.dbo.Llave_Cenco_SAE"
df_cen=get_data_sql(query, params_sa)

# query = "SELECT * FROM CRONOX.dbo.Llave_Financiera_Efe_Neg"
# df_efe_neg=get_data_sql(query, params_sa)

# query = "SELECT * FROM CRONOX.dbo.Llave_Financiera_Efe"
# df_efe_consumo=get_data_sql(query, params_sa)


In [71]:
df_cen.head()

,TITLE,FIRST NAME,LAST NAME,ADDRESS1,Address2,Address3,COMMENTS,SECURITY PHRASE,EMAIL,COMMENTS_CD,...,Mejor_Fecha_Cliente,Mejor_Duracion_Cliente,Ultimo_Descripcion_Telefono,Ultimo_Fecha_Telefono,Ultima_Duracion_Telefono,Ultimo_Descripcion_Cli,Ultimo_Fecha_Cli,Ultima_Duracion_Cli,DNI_Ejecutivo,Codigo_Paleta
0,SAE,Regimen Lab. INDEP,LIMA,CORREA LINARES CARLOS,AVENIDA PASEO DE LA REPUBLICA SAN ANTONIO 6266...,10,DEUDA_ACTUAL Sin Fecha MONTO_DESEMBOLSAR Sin M...,LINEA SAE 25000,PCT: 016 TEA: 0.099,//CONSENTIMIENTO:Y//ENTIDAD1://DEUDA1://ENTIDA...,...,2026-02-25 15:27:28,8,1.SIN GESTION,1900-01-01,0,MENSAJE EN CASILLA DE VOZ (AUTO),2026-02-27 11:43:28,0,PFC021,NaN
1,SAE,Regimen Lab. INDEP,LIMA,CORREA LINARES CARLOS,AVENIDA PASEO DE LA REPUBLICA SAN ANTONIO 6266...,10,DEUDA_ACTUAL Sin Fecha MONTO_DESEMBOLSAR Sin M...,LINEA SAE 25000,PCT: 016 TEA: 0.099,//CONSENTIMIENTO:Y//ENTIDAD1://DEUDA1://ENTIDA...,...,2026-02-25 15:27:28,8,1.SIN GESTION,1900-01-01,0,MENSAJE EN CASILLA DE VOZ (AUTO),2026-02-27 11:43:28,0,PFC021,NaN
2,SAE,Regimen Lab. INDEP,LIMA,CORREA LINARES CARLOS,AVENIDA PASEO DE LA REPUBLICA SAN ANTONIO 6266...,10,DEUDA_ACTUAL Sin Fecha MONTO_DESEMBOLSAR Sin M...,LINEA SAE 25000,PCT: 016 TEA: 0.099,//CONSENTIMIENTO:Y//ENTIDAD1://DEUDA1://ENTIDA...,...,2026-02-25 15:27:28,8,1.SIN GESTION,1900-01-01,0,MENSAJE EN CASILLA DE VOZ (AUTO),2026-02-27 11:43:28,0,PFC021,NaN
3,SAE,Regimen Lab. INDEP,LIMA,MAGUINA RODRIGUEZ DE VELASQUEZ ROSA,AVENIDA ENRIQUE MEIGGS URBANIZACION 262 URBANI...,15,DEUDA_ACTUAL Sin Fecha MONTO_DESEMBOLSAR Sin M...,LINEA SAE 1500,PCT: 035 TEA: 0.459,//CONSENTIMIENTO:Y//ENTIDAD1://DEUDA1://ENTIDA...,...,2026-02-20 11:16:14,0,1.SIN GESTION,1900-01-01,0,MENSAJE EN CASILLA DE VOZ (AUTO),2026-02-20 11:16:14,0,VDAD,NaN
4,SAE,Regimen Lab. INDEP,LIMA,MAGUINA RODRIGUEZ DE VELASQUEZ ROSA,AVENIDA ENRIQUE MEIGGS URBANIZACION 262 URBANI...,15,DEUDA_ACTUAL Sin Fecha MONTO_DESEMBOLSAR Sin M...,LINEA SAE 1500,PCT: 035 TEA: 0.459,//CONSENTIMIENTO:Y//ENTIDAD1://DEUDA1://ENTIDA...,...,2026-02-20 11:16:14,0,1.SIN GESTION,1900-01-01,0,MENSAJE EN CASILLA DE VOZ (AUTO),2026-02-20 11:16:14,0,VDAD,NaN


In [65]:
orden_01=df_efe_consumo.columns.to_list

In [57]:
df_uno=pd.read_excel("C:\\Users\\DATA\\Documents\\datos\\01_script\\inicio\\27-02 SGT.xlsx")

In [58]:
df_uno.columns = (
    df_uno.columns
    .str.strip()
    .str.upper()
    .str.replace("Á","A")
    .str.replace("É","E")
    .str.replace("Í","I")
    .str.replace("Ó","O")
    .str.replace("Ú","U")
)

In [59]:
df_uno['PHONE NUMBER'] = df_uno['NUMERO'].astype(str)


In [60]:
df_uno=df_uno[['DNI', 'NOMBRE', 'PHONE NUMBER']]

In [61]:
df_uno=df_uno.merge(df_efe_consumo, left_on='PHONE NUMBER', right_on='PHONE NUMBER', how='left')

In [ ]:
Index(['LAST NAME', 'FIRST NAME', 'ADDRESS1', 'ADDRESS3', 'COMMENTS',
       'VENDOR LEAD CODE', 'PROVINCE', 'CITY', 'EMAIL', 'COMMENTS_2',
       'PHONE NUMBER', 'Order_Numero', 'Oferta', 'Tip_Prioridad', 'MARCA_2025',
       'NOMCOMERCIAL', 'Perfil_ic', 'tipocliente', 'TIPOINGRESO', 'ASIGNACION',
       'LINEA_FT', 'LINEA_HS_RS', 'LINEA_HS_PLUS', 'LINEA_FULL', 'MARCA',
       'TIP_TELF', 'PROVEEDOR', 'PERFIL', 'SEGMENTO', 'SCORE', 'TASA',
       'RANGO_TASA', 'ZONA', 'Tipo_Telf', 'Fecha_Llam', 'Hora_Llamada',
       'FECHA_ENVIO', 'RETIRO', 'Mejor_Estado_Telefono',
       'Mejor_Sub_Estado_Telefono', 'Mejor_Descripcion_Telefono',
       'Mejor_Fecha_Telefono', 'Mejor_Hora_Telefono', 'Mejor_Durac_Telefono',
       'Mejor_Estado_Cliente', 'Mejor_Sub_Estado_Cliente',
       'Mejor_Descripcion_Cliente', 'Mejor_fecha_Cliente',
       'Mejor_Hora_Cliente', 'Mejor_Durac_Cliente', 'ULT_Descripcion_Cli',
       'ULT_fecha_Cli', 'ULT_Hora_Cli', 'ULT_Durac_Cli', 'ULT_Descripcion_Tel',
       'ULT_fecha_Tel', 'ULT_Hora_Tel', 'ULT_Durac_Tel', 'Veces_Cliente',
       'Veces_Telefono', 'REPITENCIAS', 'Cantidad_Sin_Discador',
       'Cantidad_Discador', 'TELEFO1', 'TELEFO2', 'TELEFO3', 'REGION',
       'MARCADESBASE', 'MARCA2', 'FLGSUBPROCESO_HS', 'SITUACIONLABORAL']pho

Index(['LAST NAME', 'FIRST NAME', 'ADDRESS1', 'ADDRESS3', 'COMMENTS',
       'VENDOR LEAD CODE', 'PROVINCE', 'CITY', 'EMAIL', 'COMMENTS_2',
       'PHONE NUMBER', 'Order_Numero', 'Oferta', 'Tip_Prioridad', 'MARCA_2025',
       'NOMCOMERCIAL', 'Perfil_ic', 'tipocliente', 'TIPOINGRESO', 'ASIGNACION',
       'LINEA_FT', 'LINEA_HS_RS', 'LINEA_HS_PLUS', 'LINEA_FULL', 'MARCA',
       'TIP_TELF', 'PROVEEDOR', 'PERFIL', 'SEGMENTO', 'SCORE', 'TASA',
       'RANGO_TASA', 'ZONA', 'Tipo_Telf', 'Fecha_Llam', 'Hora_Llamada',
       'FECHA_ENVIO', 'RETIRO', 'Mejor_Estado_Telefono',
       'Mejor_Sub_Estado_Telefono', 'Mejor_Descripcion_Telefono',
       'Mejor_Fecha_Telefono', 'Mejor_Hora_Telefono', 'Mejor_Durac_Telefono',
       'Mejor_Estado_Cliente', 'Mejor_Sub_Estado_Cliente',
       'Mejor_Descripcion_Cliente', 'Mejor_fecha_Cliente',
       'Mejor_Hora_Cliente', 'Mejor_Durac_Cliente', 'ULT_Descripcion_Cli',
       'ULT_fecha_Cli', 'ULT_Hora_Cli', 'ULT_Durac_Cli', 'ULT_Descripcion_Tel',
       'U

In [68]:
df_uno=df_uno[['LAST NAME', 'FIRST NAME', 'ADDRESS1', 'ADDRESS3', 'COMMENTS',
       'VENDOR LEAD CODE', 'PROVINCE', 'CITY', 'EMAIL', 'COMMENTS_2',
       'PHONE NUMBER']]

In [ ]:
df_uno

In [69]:
df_uno.to_csv("C:\\Users\\DATA\\Documents\\datos\\01_script\\inicio\\base_01.csv", index=False,sep=';')

(593162, 71)

In [ ]:
# df_efe_consumo['Mejor_Sub_Estado_Cliente'].unique()
    ['Mejor_Descripcion_Cliente'].unique()

<StringArray>
[                             'NO QUIERE',
                 'NO NECESITA / NO DESEA',
                              'SI QUIERE',
                         'ESTA ENDEUDADO',
                        'DESEA MAS MONTO',
                        'VOLVER A LLAMAR',
                                  'OTROS',
                     'CREDITO CONCRETADO',
                         'LO VA A PENSAR',
                       'CLIENTE DE VIAJE',
                     'INTERESES ELEVADOS',
        'TITULAR NO CALIFICA - DESEMPLEO',
                        'ABONO EN CUENTA',
            'CASA EN ZONA NO COBERTURADA',
                  'SOLICITA SER RETIRADO',
         'NEGOCIO EN ZONA NO COBERTURADA',
             'MALA EXPERIENCIA (TDA/FNC)',
 'TITULAR NO CALIFICA- GIRO RESTRINGIDO9',
                         'CONFIRMAR CITA',
      'TITULAR CONTAGIADO CON COVID - 19',
                          'DESEA ELECTRO',
                      'CLIENTE RECHAZADO']
Length: 22, dtype: str

In [54]:
df_efe_consumo[df_efe_consumo['VENDOR LEAD CODE']=='47905540'].head()

,LAST NAME,FIRST NAME,ADDRESS1,ADDRESS3,COMMENTS,VENDOR LEAD CODE,PROVINCE,CITY,EMAIL,COMMENTS_2,...,Cantidad_Sin_Discador,Cantidad_Discador,TELEFO1,TELEFO2,TELEFO3,REGION,MARCADESBASE,MARCA2,FLGSUBPROCESO_HS,SITUACIONLABORAL


In [10]:
df_efe_consumo.columns

Index(['LAST NAME', 'FIRST NAME', 'ADDRESS1', 'ADDRESS3', 'COMMENTS',
       'VENDOR LEAD CODE', 'PROVINCE', 'CITY', 'EMAIL', 'COMMENTS_2',
       'PHONE NUMBER', 'Order_Numero', 'Oferta', 'Tip_Prioridad', 'MARCA_2025',
       'NOMCOMERCIAL', 'Perfil_ic', 'tipocliente', 'TIPOINGRESO', 'ASIGNACION',
       'LINEA_FT', 'LINEA_HS_RS', 'LINEA_HS_PLUS', 'LINEA_FULL', 'MARCA',
       'TIP_TELF', 'PROVEEDOR', 'PERFIL', 'SEGMENTO', 'SCORE', 'TASA',
       'RANGO_TASA', 'ZONA', 'Tipo_Telf', 'Fecha_Llam', 'Hora_Llamada',
       'FECHA_ENVIO', 'RETIRO', 'Mejor_Estado_Telefono',
       'Mejor_Sub_Estado_Telefono', 'Mejor_Descripcion_Telefono',
       'Mejor_Fecha_Telefono', 'Mejor_Hora_Telefono', 'Mejor_Durac_Telefono',
       'Mejor_Estado_Cliente', 'Mejor_Sub_Estado_Cliente',
       'Mejor_Descripcion_Cliente', 'Mejor_fecha_Cliente',
       'Mejor_Hora_Cliente', 'Mejor_Durac_Cliente', 'ULT_Descripcion_Cli',
       'ULT_fecha_Cli', 'ULT_Hora_Cli', 'ULT_Durac_Cli', 'ULT_Descripcion_Tel',
       'U

# Add list

In [115]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
# from pyspark.sql.functions import last_day, col
from dateutil.relativedelta import relativedelta

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

fecha_mes_base = '2026-03-01'

query = f"""
    SELECT *
    FROM OPENQUERY([192.168.3.76], '
        SELECT        
        rtrim(ltrim(d.vendor_lead_code)) AS dni,        
        a.campaign_id AS Numero_Campana,        
        e.campaign_name AS Nombre_Campana,        
        a.user AS DNI_Ejecutivo,        
        c.full_name AS Ejecutivo,        
        a.call_date AS Fecha_Hora_Llamada,        
        a.length_in_sec AS segundos,        
        DATE(a.call_date) AS Fecha_Llamada,        
        HOUR(a.call_date) AS Trama_Hora,        
        '''' '''' AS Estados,        
        '''' '''' AS Sub_estado,        
        b.status_name AS Descripcion,        
        f.list_description,        
        f.list_name,        
        a.PHONE_NUMBER,        
        d.alt_phone as Fecha_Agenda,        
        d.comments as Comentarios,        
        a.status AS CODIGO,        
        FROM_UNIXTIME(a.start_epoch) as Inicio,        
        FROM_UNIXTIME(a.end_epoch) as Fin        
        FROM asterisk.vicidial_log a         
        LEFT JOIN asterisk.vicidial_list d ON a.lead_id=d.lead_id        
        LEFT JOIN asterisk.vicidial_campaigns e ON a.campaign_id=e.campaign_id        
        LEFT JOIN asterisk.vicidial_lists f ON a.list_id=f.list_id        
        LEFT JOIN asterisk.vicidial_statuses b ON a.status=b.status        
        LEFT JOIN asterisk.vicidial_users c ON a.user=c.user        
        WHERE call_date >= curdate()  and e.campaign_name like "%SAE"
    ')

    """
df_tmp_llamadas_cencosud=obtener_tabla_sql(spark,query,server_sa,user_sa,pwd_sa,db_sa)

In [116]:
df_tmp_llamadas_cencosud.count()

22482

In [ ]:
df_tmp_llamadas_cencosud=df_tmp_llamadas_cencosud.join(df_efe_tipif_spark,["CODIGO"],"left")

window_spec = Window.partitionBy("dni","PHONE_NUMBER").orderBy(col("PESO").asc(),col("Fecha_Llamada").desc())
df_tmp_llamadas_cencosud = df_tmp_llamadas_cencosud.withColumn("mejor_Tel", row_number().over(window_spec))

window_spec = Window.partitionBy("dni").orderBy(col("PESO").asc(),col("Fecha_Llamada").desc())
df_tmp_llamadas_cencosud = df_tmp_llamadas_cencosud.withColumn("mejor_resul", row_number().over(window_spec))

window_spec = Window.partitionBy("dni",'PHONE_NUMBER').orderBy(col("Fecha_Hora_Llamada").desc())
df_tmp_llamadas_cencosud = df_tmp_llamadas_cencosud.withColumn("ult_llamada_tel", row_number().over(window_spec))

window_spec = Window.partitionBy("dni").orderBy(col("Fecha_Hora_Llamada").desc())
df_tmp_llamadas_cencosud = df_tmp_llamadas_cencosud.withColumn("ult_llamada_cli", row_number().over(window_spec))

window_count = Window.partitionBy("dni", "PHONE_NUMBER")
df_tmp_llamadas_cencosud = df_tmp_llamadas_cencosud.withColumn(
    "nCant_agent",
    count(
        when(col("DNI_Ejecutivo") != "VDAD", 1)
    ).over(window_count)
)
df_tmp_llamadas_cencosud = df_tmp_llamadas_cencosud.withColumn(
    "nCant_no_agent",
    count(
        when(col("DNI_Ejecutivo") == "VDAD", 1)
    ).over(window_count)
)
window_count = Window.partitionBy("dni")
df_tmp_llamadas_cencosud = df_tmp_llamadas_cencosud.withColumn("q_llamada_cli",count("*").over(window_count))
window_count = Window.partitionBy("dni", "PHONE_NUMBER")
df_tmp_llamadas_cencosud = df_tmp_llamadas_cencosud.withColumn("q_llamada_cli_tel",count("*").over(window_count))

# df_tmp_llamadas_cencosud = df_tmp_llamadas_cencosud.filter(col("mejor_Tel") == 1).drop('mejor_Tel')
